#### main function

In [3]:
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, AutoConfig
import os
import glob

torch.set_grad_enabled(False)


def extract_untrained_mbert_features(
    podcast_files: list[str] = None,
    modelname: str = "bert-base-multilingual-cased",
    model_alias: str = "mbert_untrained",
    device: str = "cpu",
    output_dir: str = "mbert_untrained_features",
):
    """
    extract mbert features using untrained (randomly initialized) model weights.
    provides baseline embeddings for comparison with trained model.
    """
    
    # find files if not specified
    if podcast_files is None:
        podcast_files = sorted(glob.glob("sentences_*_timing.csv"))
    
    if len(podcast_files) == 0:
        print("no files found! please check file names and paths.")
        return
    
    print(f"found {len(podcast_files)} files to process")
    
    # load tokenizer (same as trained model)
    tokenizer = AutoTokenizer.from_pretrained(modelname)
    
    # create untrained model with random weights
    print("creating untrained bert model...")
    config = AutoConfig.from_pretrained(modelname)
    model = AutoModel.from_config(config)  # key: using from_config for random weights
    
    model = model.eval()
    model = model.to(device)
    
    # verify it's untrained
    embed_weights = model.embeddings.word_embeddings.weight.data
    print(f"weight stats - mean: {embed_weights.mean():.6f}, std: {embed_weights.std():.6f}")
    print("(should be close to mean=0, std=0.02 for untrained model)")
    
    cls_token_id = tokenizer.cls_token_id
    sep_token_id = tokenizer.sep_token_id
    
    # process each podcast file
    for podcast_file in tqdm(podcast_files, desc="processing files"):
        # extract podcast number from filename
        podcast_num = podcast_file.split('_')[-2]
        
        # read sentences
        sentence_df = pd.read_csv(podcast_file)
        print(f"\nprocessing podcast {podcast_num}: {len(sentence_df)} sentences")
        
        # create output directory
        output_path = os.path.join(output_dir, model_alias, f"podcast_{podcast_num}")
        os.makedirs(output_path, exist_ok=True)
        
        # tokenize all sentences to create flat token list
        all_tokens = []
        sentence_boundaries = []
        
        for idx, row in sentence_df.iterrows():
            start_idx = len(all_tokens)
            
            # tokenize sentence without special tokens
            token_ids = tokenizer.encode(row['text'], add_special_tokens=False)
            tokens = tokenizer.convert_ids_to_tokens(token_ids)
            
            # add to flat list
            for token_id, token in zip(token_ids, tokens):
                all_tokens.append({
                    'token_id': token_id,
                    'token': token,
                    'sentence_id': row['sentence_id'],
                    'sentence_idx': idx
                })
            
            end_idx = len(all_tokens)
            sentence_boundaries.append((start_idx, end_idx))
        
        # convert to dataframe
        token_df = pd.DataFrame(all_tokens)
        token_df['token_idx'] = range(len(token_df))
        
        # extract embeddings for each sentence with context
        all_embeddings = []
        
        for sent_idx, (start_idx, end_idx) in enumerate(tqdm(sentence_boundaries, 
                                                             desc="extracting embeddings", 
                                                             leave=False)):
            # calculate context size
            n_sentence_tokens = end_idx - start_idx
            n_context_tokens = 509 - n_sentence_tokens  # 512 - 3 special tokens
            
            # get context tokens
            context_start_idx = max(0, start_idx - n_context_tokens)
            context_tokens = token_df.iloc[context_start_idx:start_idx]['token_id'].tolist()
            sentence_tokens = token_df.iloc[start_idx:end_idx]['token_id'].tolist()
            
            # construct input: [CLS] context [SEP] sentence [SEP]
            input_ids = (
                [cls_token_id] +
                context_tokens +
                [sep_token_id] +
                sentence_tokens +
                [sep_token_id]
            )
            
            # convert to tensor
            example = torch.tensor([input_ids]).to(device)
            
            if example.shape[-1] > 512:
                print(f"error: sentence {sent_idx} exceeds 512 tokens!")
                continue
            
            # get model outputs
            with torch.no_grad():
                outputs = model(input_ids=example, output_hidden_states=True)
            
            # extract embeddings for sentence tokens only
            sentence_states = [
                state[:, -len(sentence_tokens) - 1 : -1, :]
                for state in outputs.hidden_states
            ]
            hidden_states = torch.vstack(sentence_states)
            all_embeddings.append(hidden_states.cpu().numpy())
        
        # concatenate all embeddings
        embeddings = np.concatenate(all_embeddings, axis=1)
        
        # save embeddings
        embeddings_file = os.path.join(output_path, 'embeddings.npy')
        np.save(embeddings_file, embeddings)
        print(f"saved embeddings: {embeddings_file} - shape: {embeddings.shape}")
        
        # save tokens
        token_df.to_csv(os.path.join(output_path, 'tokens.csv'), index=False)
        
        # save model info
        with open(os.path.join(output_path, 'model_info.txt'), 'w') as f:
            f.write(f"model: {modelname}\n")
            f.write("weights: untrained (random initialization)\n")
            f.write(f"embeddings shape: {embeddings.shape}\n")


def verify_untrained_vs_trained(trained_path, untrained_path, podcast_num=1):
    """
    verify that untrained embeddings are different from trained ones
    """
    # load embeddings
    trained = np.load(f"{trained_path}/mbert/podcast_{podcast_num}/embeddings.npy")
    untrained = np.load(f"{untrained_path}/mbert_untrained/podcast_{podcast_num}/embeddings.npy")
    
    # compare sample from last layer
    trained_sample = trained[12, :100, :].flatten()
    untrained_sample = untrained[12, :100, :].flatten()
    
    # cosine similarity
    cos_sim = np.dot(trained_sample, untrained_sample) / (
        np.linalg.norm(trained_sample) * np.linalg.norm(untrained_sample)
    )
    
    print(f"\nverification:")
    print(f"cosine similarity: {cos_sim:.4f} (should be close to 0)")
    print(f"trained mean: {np.mean(trained_sample):.4f}, std: {np.std(trained_sample):.4f}")
    print(f"untrained mean: {np.mean(untrained_sample):.4f}, std: {np.std(untrained_sample):.4f}")


# main execution
if __name__ == "__main__":
    device = "cpu"
    output_dir = os.getcwd()
    
    print("extracting embeddings with untrained bert model...\n")
    
    # extract features
    extract_untrained_mbert_features(
        podcast_files=None,  # auto-detect files
        modelname="bert-base-multilingual-cased",
        model_alias="mbert_untrained",
        device=device,
        output_dir=output_dir
    )
    
    print("\ndone! untrained embeddings saved.")

extracting embeddings with untrained bert model...

found 1 files to process
creating untrained bert model...
weight stats - mean: -0.000001, std: 0.020003
(should be close to mean=0, std=0.02 for untrained model)


processing files:   0%|                                   | 0/1 [00:00<?, ?it/s]


processing podcast 1: 296 sentences



processing files: 100%|███████████████████████████| 1/1 [00:19<00:00, 19.49s/it]


saved embeddings: /Users/xinyuanyan/Desktop/BCM2025/1_language/1_bilingual/study2_eatpraylove/2_mbert_epl/1_get_mbert_baseline/english/mbert_untrained/podcast_1/embeddings.npy - shape: (13, 6361, 768)

done! untrained embeddings saved.


#### load podcast 1 .. file

#### check the file

In [10]:
import numpy as np
import pandas as pd

# load the files
podcast_num = 1
base_path = f"mbert_untrained/podcast_{podcast_num}"

embeddings = np.load(f"{base_path}/embeddings.npy")
embeddings.shape

(13, 6361, 768)

In [12]:
import numpy as np
import pandas as pd

# Load files
podcast_num = 1
base_path = f"mbert_untrained/podcast_{podcast_num}"

# Load data
embeddings = np.load(f"{base_path}/embeddings.npy")  # Shape: (13, 2007, 768)
tokens_df = pd.read_csv(f"{base_path}/tokens.csv")
sentence_df = pd.read_csv(f"sentences_with_{podcast_num}_timing.csv")  # Original sentences

print(f"Embeddings shape: {embeddings.shape}")
print(f"Tokens: {len(tokens_df)} tokens")
print(f"Sentences: {len(sentence_df)} sentences")


#  accurate mapping using sentence text
def create_word_mapping_accurate():
    """Create more accurate word mapping by aligning with the original sentence"""
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
    
    word_mappings = []
    
    for _, sent_row in sentence_df.iterrows():
        sentence_id = sent_row['sentence_id']
        sentence_text = sent_row['text']
        first_word_idx = sent_row['first_word_idx']
        
        # Get sentence tokens from tokens_df
        sentence_tokens = tokens_df[tokens_df['sentence_id'] == sentence_id].sort_values('token_idx')
        
        if len(sentence_tokens) == 0:
            continue
            
        # Split sentence into words
        words = sentence_text.split()
        
        # Get token spans for each word
        current_token_pos = 0
        
        for word_pos, word in enumerate(words):
            word_idx = first_word_idx + word_pos
            
            # Tokenize this word
            word_tokens = tokenizer.tokenize(word)
            n_word_tokens = len(word_tokens)
            
            # Map tokens to this word
            for i in range(n_word_tokens):
                if current_token_pos < len(sentence_tokens):
                    token_row = sentence_tokens.iloc[current_token_pos]
                    
                    word_mappings.append({
                        'word_idx': word_idx,
                        'token_idx': token_row['token_idx'],
                        'token': token_row['token'],
                        'token_id': token_row['token_id'],
                        'sentence_id': sentence_id
                    })
                    
                    current_token_pos += 1
    
    return pd.DataFrame(word_mappings)

######### 

word_mapping_df = create_word_mapping_accurate()

# Save the mapping
mapping_file = f"{base_path}/word_token_mapping.csv"
word_mapping_df.to_csv(mapping_file, index=False)
print(f"Saved word mapping to: {mapping_file}")

# Display statistics
print(f"\nMapping statistics:")
print(f"Total mappings: {len(word_mapping_df)}")
print(f"Unique words: {word_mapping_df['word_idx'].nunique()}")
print(f"Tokens per word: {len(word_mapping_df) / word_mapping_df['word_idx'].nunique():.2f}")

# Show example mappings
print("\nExample word mappings:")
example_word = word_mapping_df['word_idx'].iloc[10]
example_mappings = word_mapping_df[word_mapping_df['word_idx'] == example_word]
print(f"Word {example_word}:")
print(example_mappings[['token_idx', 'token', 'token_id']])


Embeddings shape: (13, 6361, 768)
Tokens: 6361 tokens
Sentences: 296 sentences
Saved word mapping to: mbert_untrained/podcast_1/word_token_mapping.csv

Mapping statistics:
Total mappings: 6361
Unique words: 4855
Tokens per word: 1.31

Example word mappings:
Word 8:
    token_idx token  token_id
10         10  holy     95652


In [14]:
import numpy as np
import pandas as pd
import string

# Define punctuation to exclude
PUNCTUATION = set(string.punctuation + '–—""''')  # Include various dash and quote types

def is_punctuation_token(token):
    """Check if a token is punctuation"""
    # Remove ## prefix for subword tokens
    clean_token = token.replace('##', '')
    # Check if it's only punctuation
    return all(c in PUNCTUATION for c in clean_token)

def get_word_embeddings_all_layers(exclude_punctuation=True):
    """Extract embeddings for all words across all layers"""
    all_results = {}
    
    # First, let's check for any issues in the mapping
    print(f"\nChecking word_mapping_df for issues...")
    print(f"Null tokens: {word_mapping_df['token'].isnull().sum()}")
    print(f"Total unique words: {word_mapping_df['word_idx'].nunique()}")
    
    # Process each layer
    for layer in range(embeddings.shape[0]):
        print(f"\nProcessing layer {layer}...")
        layer_results = []
        
        for word_idx in sorted(word_mapping_df['word_idx'].unique()):
            # Get tokens for this word
            word_tokens = word_mapping_df[word_mapping_df['word_idx'] == word_idx]
            
            # Reconstruct word from tokens
            tokens = word_tokens['token'].values
            tokens_str = [str(t) if pd.notna(t) else '' for t in tokens]
            word_text = ''.join(tokens_str).replace('##', '')
            
            # Skip if word is only punctuation and exclude_punctuation is True
            if exclude_punctuation and all(c in PUNCTUATION for c in word_text):
                continue
            
            # Skip if no valid tokens
            token_indices = word_tokens['token_idx'].values
            if len(token_indices) == 0:
                continue
            
            # Filter out punctuation tokens if needed
            if exclude_punctuation:
                # Keep only non-punctuation tokens
                non_punct_mask = [not is_punctuation_token(str(t)) for t in tokens]
                if any(non_punct_mask):
                    token_indices = token_indices[non_punct_mask]
                else:
                    # All tokens are punctuation, skip this word
                    continue
            
            # Get embeddings
            token_embeddings = embeddings[layer, token_indices, :]
            word_embedding = token_embeddings.mean(axis=0)
            
            row = {
                'word_idx': word_idx,
                'word': word_text,
                'n_tokens': len(token_indices),  # Number of tokens used (after filtering)
                'sentence_id': word_tokens['sentence_id'].iloc[0]
            }
            
            # Add embedding dimensions
            for i in range(768):
                row[f'dim_{i}'] = word_embedding[i]
                
            layer_results.append(row)
        
        all_results[layer] = pd.DataFrame(layer_results)
        print(f"Layer {layer}: Extracted {len(layer_results)} words (excluded punctuation: {exclude_punctuation})")
    
    return all_results

# Extract embeddings for all layers

all_layer_embeddings = get_word_embeddings_all_layers(exclude_punctuation=True)

# Save each layer's embeddings
for layer, df in all_layer_embeddings.items():
    output_file = f"{base_path}/word_embeddings_layer{layer}.csv"
    df.to_csv(output_file, index=False)
    print(f"Saved layer {layer} embeddings to: {output_file}")



Checking word_mapping_df for issues...
Null tokens: 0
Total unique words: 4855

Processing layer 0...
Layer 0: Extracted 4855 words (excluded punctuation: True)

Processing layer 1...
Layer 1: Extracted 4855 words (excluded punctuation: True)

Processing layer 2...
Layer 2: Extracted 4855 words (excluded punctuation: True)

Processing layer 3...
Layer 3: Extracted 4855 words (excluded punctuation: True)

Processing layer 4...
Layer 4: Extracted 4855 words (excluded punctuation: True)

Processing layer 5...
Layer 5: Extracted 4855 words (excluded punctuation: True)

Processing layer 6...
Layer 6: Extracted 4855 words (excluded punctuation: True)

Processing layer 7...
Layer 7: Extracted 4855 words (excluded punctuation: True)

Processing layer 8...
Layer 8: Extracted 4855 words (excluded punctuation: True)

Processing layer 9...
Layer 9: Extracted 4855 words (excluded punctuation: True)

Processing layer 10...
Layer 10: Extracted 4855 words (excluded punctuation: True)

Processing laye